# Tarea 2 - Comunicacion cliente-servidor con A2A

En esta practica se implementa una comunicacion basica entre un cliente y un servidor siguiendo la idea del protocolo A2A. El objetivo no es montar una aplicacion grande, sino dejar claro el flujo completo: un agente se publica como servicio, otro componente lo descubre y despues le envia una peticion.

La practica toma como referencia los ejemplos oficiales de Microsoft Agent Framework. En esos ejemplos, el agente servidor publica una tarjeta de descripcion (`AgentCard`) y el cliente la utiliza para saber donde enviar los mensajes. Aqui se reproduce ese mismo esquema dentro de un notebook, para que sea facil de ejecutar y revisar paso a paso.

El flujo que se va a construir es el siguiente:

1. El servidor arranca en local y publica su `AgentCard` en `/.well-known/agent.json`.
2. El cliente consulta esa tarjeta y obtiene la informacion del agente remoto.
3. El cliente envia una pregunta mediante JSON-RPC usando el metodo `message/send`.
4. El servidor procesa el mensaje, consulta el modelo configurado en `.env` y devuelve una tarea con estado `completed`.

Referencias utilizadas:

- https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/a2a
- https://github.com/microsoft/agent-framework/tree/main/python/samples/04-hosting/a2a

## 1. Configuracion inicial

La primera parte carga las credenciales desde el archivo `.env`. En este proyecto ya estan preparadas, por lo que el notebook solo comprueba que existen las variables necesarias antes de crear el cliente del modelo.

Se usan tres valores principales:

- `AZURE_OPENAI_ENDPOINT`: endpoint del recurso de Azure OpenAI.
- `AZURE_OPENAI_API_KEY`: clave de acceso.
- `MODEL_DEPLOYMENT_NAME`: nombre del despliegue que se va a llamar.

Esta comprobacion evita que el fallo aparezca mas adelante, cuando el servidor ya este levantado.

In [ ]:
import json
import os
import threading
import time
import uuid
from dataclasses import dataclass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from typing import Any
from urllib.parse import urlparse

import httpx
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "").strip().rstrip("/") + "/"
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "").strip()
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME", "").strip()
AGENT_NAME = os.getenv("AGENT_NAME", "agente-a2a-tarea2").strip()

missing = []
if not AZURE_OPENAI_ENDPOINT or "/openai/v1/" not in AZURE_OPENAI_ENDPOINT:
    missing.append("AZURE_OPENAI_ENDPOINT con forma https://.../openai/v1")
if not AZURE_OPENAI_API_KEY or "PEGA_AQUI" in AZURE_OPENAI_API_KEY:
    missing.append("AZURE_OPENAI_API_KEY real")
if not MODEL_DEPLOYMENT_NAME:
    missing.append("MODEL_DEPLOYMENT_NAME")

if missing:
    raise RuntimeError("Faltan variables en .env: " + ", ".join(missing))

llm_client = OpenAI(base_url=AZURE_OPENAI_ENDPOINT, api_key=AZURE_OPENAI_API_KEY)

print("Configuracion cargada")
print("Endpoint:", AZURE_OPENAI_ENDPOINT)
print("Deployment:", MODEL_DEPLOYMENT_NAME)
print("Agente:", AGENT_NAME)

## 2. Estructura comun del protocolo

Antes de crear el servidor y el cliente conviene definir las estructuras que ambos van a compartir. En A2A el agente remoto se describe mediante una `AgentCard`. Esta tarjeta indica el nombre del agente, su URL, las capacidades que ofrece y el tipo de entrada y salida que acepta.

Tambien se define una funcion para extraer texto de un mensaje A2A y otra para construir la respuesta como una tarea completada. De esta forma el servidor no devuelve solo texto plano, sino una respuesta con una estructura parecida a la que se espera en una comunicacion A2A.

In [ ]:
@dataclass(frozen=True)
class A2AConfig:
    host: str = "127.0.0.1"
    port: int = 5055

    @property
    def base_url(self) -> str:
        return f"http://{self.host}:{self.port}"


a2a_config = A2AConfig()


def build_agent_card(base_url: str) -> dict[str, Any]:
    return {
        "name": AGENT_NAME,
        "description": "Agente A2A de ejemplo para la Tarea 2. Recibe preguntas por JSON-RPC y responde con Azure OpenAI.",
        "url": base_url + "/",
        "version": "1.0.0",
        "protocolVersion": "0.3.0",
        "preferredTransport": "JSONRPC",
        "capabilities": {
            "streaming": False,
            "pushNotifications": False,
            "stateTransitionHistory": False,
        },
        "defaultInputModes": ["text/plain"],
        "defaultOutputModes": ["text/plain"],
        "skills": [
            {
                "id": "responder-preguntas",
                "name": "Responder preguntas",
                "description": "Contesta al usuario usando el LLM configurado en el archivo .env.",
                "tags": ["a2a", "json-rpc", "azure-openai"],
                "examples": ["Explica en dos frases que es A2A"],
            }
        ],
    }


def extract_text_from_a2a_message(message: dict[str, Any]) -> str:
    texts: list[str] = []
    for part in message.get("parts", []):
        part_kind = part.get("kind") or part.get("type")
        if part_kind == "text" and part.get("text"):
            texts.append(str(part["text"]))
    return "\n".join(texts).strip()


def build_completed_task(user_text: str, answer_text: str) -> dict[str, Any]:
    context_id = str(uuid.uuid4())
    task_id = str(uuid.uuid4())
    response_message = {
        "kind": "message",
        "role": "agent",
        "messageId": str(uuid.uuid4()),
        "contextId": context_id,
        "parts": [{"kind": "text", "text": answer_text}],
    }
    return {
        "id": task_id,
        "contextId": context_id,
        "kind": "task",
        "status": {
            "state": "completed",
            "message": response_message,
        },
        "artifacts": [
            {
                "artifactId": str(uuid.uuid4()),
                "name": "respuesta",
                "parts": [{"kind": "text", "text": answer_text}],
            }
        ],
        "metadata": {"inputPreview": user_text[:120]},
    }


agent_card = build_agent_card(a2a_config.base_url)
agent_card

## 3. Logica interna del agente

Esta seccion contiene la parte que realmente resuelve la peticion. El servidor recibe un mensaje A2A, extrae la pregunta del usuario y llama al modelo configurado en Azure OpenAI.

La funcion `handle_jsonrpc` actua como punto de entrada para las peticiones JSON-RPC. Si el metodo recibido es `message/send`, se procesa el mensaje. Si llega otro metodo, se devuelve un error controlado. Esto permite ver con claridad donde empieza la capa de protocolo y donde empieza la logica del agente.

In [ ]:
SYSTEM_PROMPT = """
Eres un agente remoto expuesto mediante protocolo A2A.
Responde en espanol, de forma clara y breve.
Cuando te pregunten por la arquitectura, explica que el cliente descubre la AgentCard y luego llama a message/send por JSON-RPC.
""".strip()


def run_remote_agent(user_text: str) -> str:
    response = llm_client.responses.create(
        model=MODEL_DEPLOYMENT_NAME,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
        ],
        max_output_tokens=350,
    )
    return response.output_text.strip()


def handle_jsonrpc(payload: dict[str, Any]) -> dict[str, Any]:
    request_id = payload.get("id")
    method = payload.get("method")

    if method != "message/send":
        return {
            "jsonrpc": "2.0",
            "id": request_id,
            "error": {"code": -32601, "message": f"Metodo no soportado: {method}"},
        }

    params = payload.get("params", {})
    message = params.get("message", {})
    user_text = extract_text_from_a2a_message(message)
    if not user_text:
        return {
            "jsonrpc": "2.0",
            "id": request_id,
            "error": {"code": -32602, "message": "El mensaje A2A no contiene texto."},
        }

    answer = run_remote_agent(user_text)
    return {"jsonrpc": "2.0", "id": request_id, "result": build_completed_task(user_text, answer)}

## 4. Servidor A2A

El servidor se ejecuta en local usando HTTP. Para esta practica se implementan dos rutas, que son las importantes para demostrar la comunicacion:

- `GET /.well-known/agent.json`: devuelve la tarjeta publica del agente. El cliente la usa para descubrir el servicio.
- `POST /`: recibe una peticion JSON-RPC con el metodo `message/send` y devuelve la respuesta del agente.

El servidor se lanza en un hilo secundario para poder seguir usando el notebook mientras queda escuchando peticiones.

In [ ]:
class A2ARequestHandler(BaseHTTPRequestHandler):
    server_version = "Tarea2A2A/1.0"

    def log_message(self, format: str, *args: Any) -> None:
        return

    def _send_json(self, status_code: int, payload: dict[str, Any]) -> None:
        body = json.dumps(payload, ensure_ascii=False, indent=2).encode("utf-8")
        self.send_response(status_code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self) -> None:
        path = urlparse(self.path).path
        if path == "/.well-known/agent.json":
            self._send_json(200, build_agent_card(a2a_config.base_url))
            return
        self._send_json(404, {"error": "Endpoint no encontrado"})

    def do_POST(self) -> None:
        path = urlparse(self.path).path
        if path != "/":
            self._send_json(404, {"error": "Endpoint no encontrado"})
            return

        try:
            content_length = int(self.headers.get("Content-Length", "0"))
            raw_body = self.rfile.read(content_length).decode("utf-8")
            payload = json.loads(raw_body)
            response = handle_jsonrpc(payload)
            status_code = 200 if "result" in response else 400
            self._send_json(status_code, response)
        except Exception as exc:
            self._send_json(
                500,
                {
                    "jsonrpc": "2.0",
                    "id": None,
                    "error": {"code": -32000, "message": type(exc).__name__, "data": str(exc)},
                },
            )


def start_a2a_server() -> ThreadingHTTPServer:
    server = ThreadingHTTPServer((a2a_config.host, a2a_config.port), A2ARequestHandler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    return server


if "a2a_server" in globals():
    try:
        a2a_server.shutdown()
        a2a_server.server_close()
    except Exception:
        pass

a2a_server = start_a2a_server()
time.sleep(0.3)

print(f"Servidor A2A activo en {a2a_config.base_url}")
print(f"AgentCard: {a2a_config.base_url}/.well-known/agent.json")

## 5. Cliente A2A

El cliente representa la otra parte de la comunicacion. Primero hace una peticion `GET` a la tarjeta del agente y, una vez que sabe donde esta el servidor, envia una pregunta con JSON-RPC.

La clase `A2AClient` separa las dos operaciones principales: descubrir el agente y enviar mensajes. Asi se puede ver de forma ordenada que la comunicacion no empieza directamente con una pregunta, sino con una fase previa de descubrimiento.

In [ ]:
class A2AClient:
    def __init__(self, base_url: str) -> None:
        self.base_url = base_url.rstrip("/")

    def get_agent_card(self) -> dict[str, Any]:
        response = httpx.get(f"{self.base_url}/.well-known/agent.json", timeout=10)
        response.raise_for_status()
        return response.json()

    def send_message(self, text: str) -> dict[str, Any]:
        payload = {
            "jsonrpc": "2.0",
            "id": str(uuid.uuid4()),
            "method": "message/send",
            "params": {
                "message": {
                    "kind": "message",
                    "role": "user",
                    "messageId": str(uuid.uuid4()),
                    "parts": [{"kind": "text", "text": text}],
                }
            },
        }
        response = httpx.post(f"{self.base_url}/", json=payload, timeout=60)
        response.raise_for_status()
        body = response.json()
        if "error" in body:
            raise RuntimeError(body["error"])
        return body["result"]

    @staticmethod
    def extract_answer(task: dict[str, Any]) -> str:
        status_message = task.get("status", {}).get("message", {})
        text = extract_text_from_a2a_message(status_message)
        if text:
            return text
        for artifact in task.get("artifacts", []):
            text = extract_text_from_a2a_message(artifact)
            if text:
                return text
        return ""


a2a_client = A2AClient(a2a_config.base_url)
discovered_card = a2a_client.get_agent_card()

print("Agente descubierto:", discovered_card["name"])
print("Transporte preferido:", discovered_card["preferredTransport"])
print("URL JSON-RPC:", discovered_card["url"])

## 6. Prueba de funcionamiento

En esta prueba se ejecuta el recorrido completo. El cliente envia una pregunta al servidor, el servidor la transforma en una llamada al modelo y finalmente devuelve una tarea A2A con la respuesta.

Si la celda termina mostrando el estado `completed`, significa que la comunicacion cliente-servidor ha funcionado correctamente.

In [ ]:
question = "Explica en 4 frases como funciona esta comunicacion cliente-servidor con A2A."

task = a2a_client.send_message(question)
answer = a2a_client.extract_answer(task)

print("Pregunta del cliente:")
print(question)
print("\nEstado de la tarea A2A:", task["status"]["state"])
print("Task ID:", task["id"])
print("\nRespuesta del agente remoto:")
print(answer)

## 7. Demo para grabar en video

Esta celda esta pensada para grabar la practica. En vez de mostrar solo el JSON, presenta la comunicacion como una conversacion entre el cliente y el agente servidor.

Cada turno sigue el mismo recorrido: el cliente envia el mensaje por A2A, el servidor lo procesa y el agente devuelve una respuesta. Las pausas cortas ayudan a que el flujo se vea claro en pantalla.

In [ ]:
from IPython.display import Markdown, display


def show_turn(sender: str, text: str) -> None:
    display(Markdown(f"**{sender}:** {text}"))
    time.sleep(0.7)


video_questions = [
    "Hola, soy el cliente. Puedes presentarte como agente A2A en una frase?",
    "Explica que ha pasado cuando he consultado tu AgentCard.",
    "Ahora responde como servidor: que haces cuando recibes un message/send?",
]

display(Markdown("### Conversacion cliente-servidor A2A"))
show_turn("Cliente", "Primero descubro el agente consultando `/.well-known/agent.json`.")
show_turn("Servidor", f"AgentCard publicada. Agente disponible: `{discovered_card['name']}`.")

conversation_log = []
for question in video_questions:
    show_turn("Cliente", question)
    task = a2a_client.send_message(question)
    answer = a2a_client.extract_answer(task)
    show_turn("Servidor / Agente", answer)
    conversation_log.append(
        {
            "cliente": question,
            "servidor": answer,
            "estado_tarea": task["status"]["state"],
            "task_id": task["id"],
        }
    )

show_turn("Cliente", "Fin de la prueba. La comunicacion A2A ha quedado completada.")

## 8. Revision de la respuesta

La siguiente celda muestra el JSON completo que devuelve el servidor. Esto es util para comprobar que la respuesta no es solo una cadena de texto, sino una estructura con identificador de tarea, estado, mensaje y artefactos.

En una defensa practica, esta salida sirve para senalar donde aparece el estado `completed` y donde se encuentra el texto generado por el agente.

In [ ]:
print(json.dumps(task, ensure_ascii=False, indent=2))

## 9. Cierre del servidor

Cuando ya se ha terminado la prueba se puede detener el servidor para liberar el puerto. La celda esta comentada para evitar cerrarlo por accidente durante la ejecucion del notebook.

In [ ]:
# a2a_server.shutdown()
# a2a_server.server_close()
# print("Servidor A2A detenido")

## Conclusiones

Con este notebook se ha construido una comunicacion cliente-servidor sencilla siguiendo el patron A2A. El servidor publica una tarjeta de agente en `/.well-known/agent.json`, el cliente la consulta para descubrir el servicio y despues envia una peticion JSON-RPC con `message/send`.

La parte importante de la practica es que el cliente no llama directamente al modelo. El cliente solo habla con el servidor A2A. Es el servidor quien recibe el mensaje, ejecuta la logica del agente y devuelve una tarea con estado `completed`.

De esta forma queda separada la responsabilidad de cada componente: el cliente se encarga de descubrir y enviar mensajes, mientras que el servidor se encarga de exponer el agente y procesar las peticiones.